<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/0_fastmcp_server_practice_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastMCP Server 실습 예제 (Colab)

이 노트북은 **FastMCP Server** 문서의 핵심 내용을 직접 실습하기 위한 예제입니다.

실습 목표:
- `FastMCP("MyServer", instructions=...)` 형태로 서버를 생성한다.
- `@mcp.tool`, `@mcp.resource`, `@mcp.prompt` 로 컴포넌트를 등록한다.
- in-memory client로 서버를 테스트한다.
- `mcp.run(transport="http", ...)` 과 `@mcp.custom_route(...)` 를 이용한 HTTP 서버 예제를 확인한다.


## 실행 순서

1. **FastMCP 설치**
2. **기본 import**
3. **FastMCP 서버 정의**
4. **in-memory client로 tool / resource / prompt 테스트**
5. **standalone server 파일 생성**
6. **HTTP 서버 실행**
7. **`/health` custom route 확인**
8. **서버 종료**




In [1]:
# 1) 설치
!pip -q install -U fastmcp requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# 2) 기본 import
from fastmcp import FastMCP, Client
from fastmcp.prompts import Message
from pathlib import Path
import json
import os
import sys
import time
import socket
import subprocess
import requests

print("import 완료")


import 완료


## 1) FastMCP 서버 정의

문서에 따르면 `FastMCP` 는 MCP 앱의 중심 컨테이너이고, 가장 단순하게는 이름만으로 생성할 수 있습니다.
`instructions` 를 주면 클라이언트와 LLM이 서버의 목적을 이해하는 데 도움이 됩니다.


In [3]:
mcp = FastMCP(
    "ColabDemoServer",
    instructions="Provides demo tools, resources, and prompts for FastMCP server practice."
)

@mcp.tool(tags={"public", "utility"})
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더합니다."""
    return a + b

@mcp.resource("resource://course/info", tags={"public", "docs"})
def course_info() -> dict:
    """실습 안내 정보를 제공합니다."""
    return {
        "course": "FastMCP Server",
        "topic": "tools, resources, prompts, custom routes",
        "mode": "colab-demo"
    }

@mcp.prompt
def explain_server(topic: str) -> list[Message]:
    """특정 주제를 설명하도록 LLM에 지시하는 프롬프트를 생성합니다."""
    return [
        Message(f"Explain the FastMCP server topic '{topic}' in concise Korean with one example.")
    ]

print("FastMCP 서버 정의 완료")


FastMCP 서버 정의 완료


## 2) in-memory client로 테스트

Colab에서는 별도 프로세스를 띄우지 않고도 `Client(mcp)` 형태로 바로 서버를 테스트할 수 있습니다.
이 방식은 tool / resource / prompt를 빠르게 검증할 때 편리합니다.


In [4]:
def obj_to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if isinstance(obj, dict):
        return obj
    result = {}
    for key in ["name", "description", "inputSchema", "input_schema", "uri", "title"]:
        if hasattr(obj, key):
            result[key] = getattr(obj, key)
    return result

def prompt_messages_to_dict(prompt_result):
    rows = []
    for msg in prompt_result.messages:
        role = getattr(msg, "role", None)
        content = getattr(msg, "content", None)
        text = getattr(content, "text", str(content))
        rows.append({"role": role, "content": text})
    return rows

def resource_items_to_text(items):
    out = []
    for item in items:
        if hasattr(item, "text"):
            out.append({"mimeType": getattr(item, "mimeType", None), "text": item.text})
        else:
            out.append(str(item))
    return out

print("helper 준비 완료")


helper 준비 완료


In [5]:
async def run_inmemory_demo():
    client = Client(mcp)

    async with client:
        print("=== list_tools ===")
        tools = await client.list_tools()
        for t in tools:
            print(json.dumps(obj_to_dict(t), ensure_ascii=False, indent=2, default=str))

        print("\n=== call_tool(add_numbers) ===")
        tool_result = await client.call_tool("add_numbers", {"a": 7, "b": 8})
        print("data:", getattr(tool_result, "data", None))
        print("content:", getattr(tool_result, "content", None))

        print("\n=== list_resources ===")
        resources = await client.list_resources()
        for r in resources:
            print(json.dumps(obj_to_dict(r), ensure_ascii=False, indent=2, default=str))

        print("\n=== read_resource(resource://course/info) ===")
        resource_result = await client.read_resource("resource://course/info")
        print(json.dumps(resource_items_to_text(resource_result), ensure_ascii=False, indent=2))

        print("\n=== list_prompts ===")
        prompts = await client.list_prompts()
        for p in prompts:
            print(json.dumps(obj_to_dict(p), ensure_ascii=False, indent=2, default=str))

        print("\n=== get_prompt(explain_server) ===")
        prompt_result = await client.get_prompt("explain_server", {"topic": "custom routes"})
        print(json.dumps(prompt_messages_to_dict(prompt_result), ensure_ascii=False, indent=2))

await run_inmemory_demo()


=== list_tools ===
{
  "name": "add_numbers",
  "title": null,
  "description": "두 정수를 더합니다.",
  "inputSchema": {
    "additionalProperties": false,
    "properties": {
      "a": {
        "type": "integer"
      },
      "b": {
        "type": "integer"
      }
    },
    "required": [
      "a",
      "b"
    ],
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "type": "integer"
      }
    },
    "required": [
      "result"
    ],
    "type": "object",
    "x-fastmcp-wrap-result": true
  },
  "icons": null,
  "annotations": null,
  "meta": {
    "fastmcp": {
      "tags": [
        "public",
        "utility"
      ]
    }
  },
  "execution": null
}

=== call_tool(add_numbers) ===
data: 15
content: [TextContent(type='text', text='15', annotations=None, meta=None)]

=== list_resources ===
{
  "name": "course_info",
  "title": null,
  "uri": "resource://course/info",
  "description": "실습 안내 정보를 제공합니다.",
  "mimeType": "text/plain",
  "size": 

## 3) standalone server 파일 생성

문서에서는 `if __name__ == "__main__": mcp.run()` 형태를 권장합니다.
아래 파일은 별도 프로세스로 실행할 수 있는 HTTP 서버 예제입니다.

추가로 `@mcp.custom_route("/health")` 를 넣어 health check endpoint도 함께 제공합니다.


In [6]:
SERVER_CODE = r"""
from fastmcp import FastMCP
from fastmcp.prompts import Message
from starlette.requests import Request
from starlette.responses import PlainTextResponse

mcp = FastMCP(
    "ColabHTTPDemo",
    instructions="Demo FastMCP server with tools, resources, prompts, and custom route."
)

@mcp.tool(tags={"public", "utility"})
def multiply(a: float, b: float) -> float:
    \"\"\"두 수를 곱합니다.\"\"\"
    return a * b

@mcp.resource("resource://server/config", tags={"public", "config"})
def get_config() -> dict:
    \"\"\"서버 설정 정보를 제공합니다.\"\"\"
    return {
        "server_name": "ColabHTTPDemo",
        "transport": "http",
        "purpose": "FastMCP server page practice"
    }

@mcp.prompt
def analyze_data(data_points: list[float]) -> str:
    \"\"\"데이터 포인트 분석용 프롬프트를 생성합니다.\"\"\"
    formatted_data = ", ".join(str(x) for x in data_points)
    return f"Please analyze these data points: {formatted_data}"

@mcp.custom_route("/health", methods=["GET"])
async def health_check(request: Request) -> PlainTextResponse:
    return PlainTextResponse("OK")

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8010)
"""
server_file = Path("fastmcp_server_demo.py")
server_file.write_text(SERVER_CODE, encoding="utf-8")
print(f"server file saved: {server_file.resolve()}")


server file saved: /content/fastmcp_server_demo.py


## 4) HTTP 서버 실행

이제 별도 프로세스로 서버를 띄워서, custom route인 `/health` 를 호출해 봅니다.


In [7]:
PORT = 8010
server_proc = subprocess.Popen(
    [sys.executable, "fastmcp_server_demo.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"server pid={server_proc.pid}")


server pid=2439


In [8]:
def wait_for_port(host: str, port: int, timeout: float = 20.0):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.2)
    return False

ready = wait_for_port("127.0.0.1", PORT, timeout=20.0)
print("HTTP 서버 준비 상태:", ready)

if not ready:
    if server_proc.poll() is not None and server_proc.stdout is not None:
        print(server_proc.stdout.read())
    raise RuntimeError("서버가 제시간에 시작되지 않았습니다.")


HTTP 서버 준비 상태: False
  File "/content/fastmcp_server_demo.py", line 14
    \"\"\"두 수를 곱합니다.\"\"\"
     ^
SyntaxError: unexpected character after line continuation character



RuntimeError: 서버가 제시간에 시작되지 않았습니다.

## 5) custom route 확인

문서의 예시처럼 HTTP transport에서는 MCP endpoint와 함께 custom route를 둘 수 있습니다.
여기서는 `/health` 가 정상 응답하는지 확인합니다.


In [ ]:
resp = requests.get("http://127.0.0.1:8010/health", timeout=10)
print("status_code:", resp.status_code)
print("text:", resp.text)


## 6) 서버 종료


In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=5)

print("서버 종료 완료")


## 실습 정리

이번 실습에서 확인한 핵심:

1. `FastMCP(...)` 는 서버의 중심 컨테이너다.
2. 서버는 **tools / resources / prompts** 를 노출할 수 있다.
3. 가장 단순한 테스트는 `Client(mcp)` 를 이용한 in-memory client로 할 수 있다.
4. 실제 실행 시에는 `if __name__ == "__main__": mcp.run()` 패턴을 사용한다.
5. HTTP transport에서는 `@mcp.custom_route(...)` 로 health check 같은 웹 경로를 추가할 수 있다.

추가 확장 아이디어:
- `strict_input_validation=True` 적용
- `mask_error_details=True` 적용
- tag filtering (`enable`, `disable`) 실습
- middleware / auth / custom routes 확장
